In [1]:
import os
import requests
import xml.etree.ElementTree as ET
from dotenv import load_dotenv

from langchain.tools import tool
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

# ---------- 1. 加载环境变量 ----------
load_dotenv()

True

In [62]:
# ---------- 2. 定义 PubMed 搜索工具 ----------
@tool
def search_pubmed(query: str, max_results: int = 5) -> str:
    """
    在 PubMed 上搜索最新的文献。
    输入一个搜索关键词（如 'CRISPR off-target 2024'），
    返回最多 max_results 篇文献的标题、作者、发表年份和摘要。
    """
    # 第一步：用 ESearch 拿到文献的 PMID 列表
    esearch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    esearch_params = {
        "db": "pubmed",
        "term": query,
        "retmax": max_results,
        "retmode": "json",
        "sort": "pub_date",  # 按发表时间倒序，拿最新的
    }
    esearch_resp = requests.get(esearch_url, params=esearch_params, timeout=30)
    esearch_data = esearch_resp.json()
    pmids = esearch_data.get("esearchresult", {}).get("idlist", [])

    if not pmids:
        return f"没有找到关于 '{query}' 的文献。"

    # 第二步：用 EFetch 根据 PMID 拿到详细信息（XML 格式）
    efetch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    efetch_params = {
        "db": "pubmed",
        "id": ",".join(pmids),
        "retmode": "xml",
    }
    efetch_resp = requests.get(efetch_url, params=efetch_params, timeout=30)
    xml_text = efetch_resp.text

    # 第三步：解析 XML，提取标题、年份、摘要
    root = ET.fromstring(xml_text)
    results = []
    for article in root.findall(".//PubmedArticle"):
        title_el = article.find(".//ArticleTitle")
        title = title_el.text if title_el is not None else "无标题"

        year_el = article.find(".//PubDate/Year")
        if year_el is None:
            year_el = article.find(".//PubDate/MedlineDate")
        year = year_el.text if year_el is not None else "未知年份"

        # 摘要在 AbstractText 里，可能有多段
        abstract_parts = article.findall(".//Abstract/AbstractText")
        abstract = " ".join([t.text for t in abstract_parts if t.text]) if abstract_parts else "无摘要"

        # 截断摘要，避免太长撑爆上下文
        if len(abstract) > 800:
            abstract = abstract[:800] + "..."

        results.append(f"【{year}】{title}\n摘要：{abstract}\n")

    return "\n---\n".join(results)

In [63]:
# ---------- 3. 初始化 LLM（用 DeepSeek） ----------
llm = ChatOpenAI(
    model="deepseek-chat",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com",
    temperature=0,
)

# ---------- 4. 创建 Agent ----------
agent = create_agent(
    model=llm,
    tools=[search_pubmed],
    system_prompt=(
        "你是一个生物学文献助手。"
        "当用户询问最新文献或某个研究方向的进展时，"
        "你必须调用 search_pubmed 工具去检索真实数据，"
        "然后基于检索结果用中文总结，不要编造内容。"
    ),
)

In [29]:
print(search_pubmed.invoke({"query": "CRISPR off-target 2024", "max_results": 5}))

[DEBUG] query='CRISPR off-target 2024', max_results=5, type=<class 'int'>
【2026】abCRISPR: deep learning-based design of abasic gRNA sequences for specific CRISPR-Cas9 genome editing.
摘要：CRISPR-Cas9 has become a widely used tool for genome editing. However, its off-target cleavage caused by partial sequence matches with guide RNAs (gRNAs) remains a critical limitation. Recently, abasic gRNAs (ØXØ) have been developed to enhance target specificity, but their effects vary depending on the positional sequence context. Here, we present abCRISPR, a deep neural network (DNN) framework for the rational design of ØXØ sequences with minimized off-target activity. abCRISPR leverages informative few-shot training with paired datasets of abasic and unmodified gRNAs, using high-quality random mismatch target libraries, exhaustively sequenced for mismatched off-target substrates (n = 97583) in in vitro CRISPR-Cas9 cleavage experiments. Predicted off-target activities for both abasic and...

---
【2026

In [64]:
# ---------- 5. 发起提问 ----------
result = agent.invoke({
    "messages": [
        {"role": "user", "content": "帮我总结一下 2024 年关于 CRISPR 脱靶效应的文章，列出 3-5 篇。"}
    ]
})

# 打印最终回答
print(result["messages"][-1].content)

我检索了 PubMed，但需要先说明一个重要问题：**检索结果中并没有 2024 年发表的文献**，返回的 5 篇标注年份均为 **2026 年**。因此我无法按你的要求总结"2024 年"的文章——如果强行归类会构成编造。

以下是检索到的实际结果（均为 2026 年，且只有部分与脱靶效应直接相关）：

**1. abCRISPR：基于深度学习的脱靶最小化 gRNA 设计**
- 直接相关。针对 CRISPR-Cas9 因 gRNA 部分序列匹配导致的脱靶切割问题，提出用深度神经网络设计无碱基 gRNA（ØXØ）序列。基于体外切割实验、对错配脱靶底物进行穷尽测序（n=97,583）构建配对数据集，用少样本学习预测脱靶活性。

**2. CRISPR-Cas9 上调 utrophin 治疗杜氏肌营养不良**
- 部分相关。通过破坏阻遏物结合位点（如 Let-7c 结合位点）上调 UTRN 表达，文中报告"脱靶效应极小"，但脱靶并非研究主题。

**3. 鸡基因组编辑效率改进（RNP 介导递送）**
- 部分相关。比较质粒与核糖核蛋白（RNP）递送方式，指出质粒递送存在脱靶效应增加、细胞毒性等问题，RNP 递送可改善。

**4. 通过病毒样颗粒（VLP）非侵入递送 CRISPR RNP 的小鼠模型构建方案**
- 部分相关。强调该方法可实现高效、可遗传的突变且"脱靶效应最小化"。

**5. Eve Smith 小说中的生殖未来**
- 不相关（文学评论，涉及生殖系编辑的伦理讨论）。

---

**建议**：如果你确实需要 2024 年的文献，我可以换用更精确的检索词（例如加上具体年份限定、或聚焦"off-target detection""off-target prediction"等子方向）重新检索。你希望我：

1. 用 `CRISPR off-target 2024` 或类似关键词再检索一次？
2. 还是聚焦某个具体方向（如脱靶检测方法、预测算法、碱基编辑脱靶等）？

请告诉我，我再为你检索真实数据。


In [7]:
esearch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
esearch_params = {
        "db": "pubmed",
        "term": 'CRISPR off-target 2024',
        "retmax": 5,
        "retmode": "json",
        "sort": "pub_date",  # 按发表时间倒序，拿最新的
    }
esearch_resp = requests.get(esearch_url, params=esearch_params, timeout=30)

In [8]:
esearch_data = esearch_resp.json()
pmids = esearch_data.get("esearchresult", {}).get("idlist", [])

In [9]:
pmids

['42525385', '41877484', '42371233', '42290533', '42192532']

In [10]:
efetch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
efetch_params = {
        "db": "pubmed",
        "id": ",".join(pmids),
        "retmode": "xml",
    }
efetch_resp = requests.get(efetch_url, params=efetch_params, timeout=30)
xml_text = efetch_resp.text

In [13]:
xml_text

'<?xml version="1.0" ?>\n<!DOCTYPE PubmedArticleSet PUBLIC "-//NLM//DTD PubMedArticle, 1st January 2025//EN" "https://dtd.nlm.nih.gov/ncbi/pubmed/out/pubmed_250101.dtd">\n<PubmedArticleSet>\n<PubmedArticle><MedlineCitation Status="MEDLINE" Owner="NLM" IndexingMethod="Automated"><PMID Version="1">42525385</PMID><DateCompleted><Year>2026</Year><Month>08</Month><Day>12</Day></DateCompleted><DateRevised><Year>2026</Year><Month>09</Month><Day>07</Day></DateRevised><Article PubModel="Print"><Journal><ISSN IssnType="Electronic">1367-4811</ISSN><JournalIssue CitedMedium="Internet"><Volume>42</Volume><Issue>8</Issue><PubDate><Year>2026</Year><Month>Aug</Month><Day>03</Day></PubDate></JournalIssue><Title>Bioinformatics (Oxford, England)</Title><ISOAbbreviation>Bioinformatics</ISOAbbreviation></Journal><ArticleTitle>abCRISPR: deep learning-based design of abasic gRNA sequences for specific CRISPR-Cas9 genome editing.</ArticleTitle><ELocationID EIdType="pii" ValidYN="Y">btag568</ELocationID><ELoca

In [15]:
root = ET.fromstring(xml_text)
root

<Element 'PubmedArticleSet' at 0x7beefcb5c220>

In [16]:
results = []
for article in root.findall(".//PubmedArticle"):
        title_el = article.find(".//ArticleTitle")
        title = title_el.text if title_el is not None else "无标题"

        year_el = article.find(".//PubDate/Year")
        if year_el is None:
            year_el = article.find(".//PubDate/MedlineDate")
        year = year_el.text if year_el is not None else "未知年份"

        # 摘要在 AbstractText 里，可能有多段
        abstract_parts = article.findall(".//Abstract/AbstractText")
        abstract = " ".join([t.text for t in abstract_parts if t.text]) if abstract_parts else "无摘要"

        # 截断摘要，避免太长撑爆上下文
        if len(abstract) > 800:
            abstract = abstract[:800] + "..."

        results.append(f"【{year}】{title}\n摘要：{abstract}\n")

In [17]:
results

['【2026】abCRISPR: deep learning-based design of abasic gRNA sequences for specific CRISPR-Cas9 genome editing.\n摘要：CRISPR-Cas9 has become a widely used tool for genome editing. However, its off-target cleavage caused by partial sequence matches with guide RNAs (gRNAs) remains a critical limitation. Recently, abasic gRNAs (ØXØ) have been developed to enhance target specificity, but their effects vary depending on the positional sequence context. Here, we present abCRISPR, a deep neural network (DNN) framework for the rational design of ØXØ sequences with minimized off-target activity. abCRISPR leverages informative few-shot training with paired datasets of abasic and unmodified gRNAs, using high-quality random mismatch target libraries, exhaustively sequenced for mismatched off-target substrates (n\u2009=\u200997583) in in vitro CRISPR-Cas9 cleavage experiments. Predicted off-target activities for both abasic and...\n',
 '【2026】CRISPR-Cas9-mediated upregulation of utrophin ameliorates D

In [34]:
# 模拟 Agent 可能传进来的参数（带换行、特殊字符）
test_query = "CRISPR off-target 2024 review\n\nPlease search"
print(search_pubmed.invoke({"query": test_query, "max_results": "5"}))

[DEBUG] query='CRISPR off-target 2024 review\n\nPlease search', max_results=5, type=<class 'int'>
[DEBUG] query bytes: b'CRISPR off-target 2024 review\n\nPlease search'
[DEBUG] query repr: 'CRISPR off-target 2024 review\n\nPlease search'
没有找到关于 'CRISPR off-target 2024 review

Please search' 的文献。
